### Recording Session

In [3]:
import time
import random
import os
from datetime import datetime

import numpy as np
from tqdm import tqdm

from brainflow.board_shim import BoardShim, BrainFlowInputParams
from brainflow.data_filter import DataFilter, FilterTypes, NoiseTypes


BOARD_ID = 0
SERIAL_PORT = "COM3"
FS = 250

NUM_TRIALS_PER_CLASS = 15

FIXATION_SEC = 2
IMAGERY_SEC = 5
REST_SEC = 3
WARMUP_SEC = 5

CSP_START_AFTER_CUE = 1.0
CSP_END_AFTER_CUE = 4.0

SAVE_DIR = "Datasets/Raw"

MARKERS = {
    "LEFT": 1,
    "RIGHT": 2,
}


def clean_name(name):
    return "".join(c for c in name.strip().upper() if c.isalnum() or c in "_-")


def get_next_session_name(subject_folder, subject_code):
    existing = [
        f for f in os.listdir(subject_folder)
        if f.startswith(subject_code + "_") and f.endswith(".npz")
    ]

    nums = []
    for f in existing:
        try:
            nums.append(int(f.replace(".npz", "").split("_")[-1]))
        except ValueError:
            pass

    next_num = max(nums, default=0) + 1
    return f"{subject_code}_{next_num:02d}"


def main():
    subject_name = input("Subject name: ").strip()
    age = input("Age: ").strip()
    gender = input("Gender: ").strip()
    mood = input("Mood / condition today: ").strip()

    subject_code = clean_name(subject_name)
    subject_folder = os.path.join(SAVE_DIR, subject_code)
    os.makedirs(subject_folder, exist_ok=True)

    session_name = get_next_session_name(subject_folder, subject_code)

    metadata = {
        "subject_name": subject_name,
        "subject_code": subject_code,
        "age": age,
        "gender": gender,
        "mood": mood,
        "session_name": session_name,
        "date_time": datetime.now().isoformat(),
        "task": "left_right_motor_imagery",
        "fixation_sec": FIXATION_SEC,
        "imagery_sec": IMAGERY_SEC,
        "rest_sec": REST_SEC,
        "warmup_sec": WARMUP_SEC,
        "num_trials_per_class": NUM_TRIALS_PER_CLASS,
        "fs": FS,
        "board_id": BOARD_ID,
        "serial_port": SERIAL_PORT,
        "csp_start_after_cue": CSP_START_AFTER_CUE,
        "csp_end_after_cue": CSP_END_AFTER_CUE,
    }

    trial_labels = (
        [MARKERS["LEFT"]] * NUM_TRIALS_PER_CLASS +
        [MARKERS["RIGHT"]] * NUM_TRIALS_PER_CLASS
    )

    random.shuffle(trial_labels)

    params = BrainFlowInputParams()
    params.serial_port = SERIAL_PORT

    BoardShim.enable_dev_board_logger()
    board = BoardShim(BOARD_ID, params)

    try:
        print("\nPreparing OpenBCI session...")
        board.prepare_session()
        board.start_stream()

        print(f"Warming up for {WARMUP_SEC}s...")
        time.sleep(WARMUP_SEC)

        print("\nStarting recording.")
        print(f"Session: {session_name}")
        print("Fixation -> cue/imagery -> rest")
        print("Do not physically move.\n")

        for trial_idx, label in enumerate(trial_labels, start=1):
            label_name = "LEFT" if label == MARKERS["LEFT"] else "RIGHT"

            print("=" * 40)
            print(f"Trial {trial_idx}/{len(trial_labels)}")
            print(f"Fixation: {FIXATION_SEC}s")
            time.sleep(FIXATION_SEC)

            print(f">>> THINK {label_name} <<<")
            board.insert_marker(label)
            time.sleep(IMAGERY_SEC)

            print(f"Rest: {REST_SEC}s")
            time.sleep(REST_SEC)

        time.sleep(1)

        print("\nStopping stream...")
        data = board.get_board_data()

    except KeyboardInterrupt:
        print("\nInterrupted. Saving collected data...")
        time.sleep(1)
        data = board.get_board_data()

    finally:
        try:
            board.stop_stream()
            board.release_session()
        except Exception:
            pass

    eeg_channels = BoardShim.get_eeg_channels(BOARD_ID)
    marker_channel = BoardShim.get_marker_channel(BOARD_ID)

    eeg_raw = data[eeg_channels]
    markers = data[marker_channel]

    eeg_filtered = eeg_raw.copy()

    print("\nFiltering...")
    for ch in tqdm(range(eeg_filtered.shape[0])):
        DataFilter.perform_bandpass(
            eeg_filtered[ch],
            FS,
            0.5,
            45.0,
            4,
            FilterTypes.BUTTERWORTH_ZERO_PHASE.value,
            0
        )

        DataFilter.remove_environmental_noise(
            eeg_filtered[ch],
            FS,
            NoiseTypes.FIFTY.value
        )

    marker_indices = np.where(markers > 0)[0]
    marker_values = markers[marker_indices].astype(int)

    pre_samp = int(FIXATION_SEC * FS)
    post_samp = int(IMAGERY_SEC * FS)

    X_raw = []
    X_filtered = []
    y = []
    kept_marker_indices = []

    for idx, label in zip(marker_indices, marker_values):
        if label not in [MARKERS["LEFT"], MARKERS["RIGHT"]]:
            continue

        start_idx = idx - pre_samp
        end_idx = idx + post_samp

        if start_idx < 0 or end_idx > eeg_raw.shape[1]:
            continue

        X_raw.append(eeg_raw[:, start_idx:end_idx].copy())
        X_filtered.append(eeg_filtered[:, start_idx:end_idx].copy())
        y.append(label)
        kept_marker_indices.append(idx)

    X_raw = np.array(X_raw)
    X_filtered = np.array(X_filtered)
    y = np.array(y)
    kept_marker_indices = np.array(kept_marker_indices)

    csp_start = int((FIXATION_SEC + CSP_START_AFTER_CUE) * FS)
    csp_end = int((FIXATION_SEC + CSP_END_AFTER_CUE) * FS)

    X_csp = X_filtered[:, :, csp_start:csp_end]

    print("\nFinal shapes:")
    print("Continuous raw:", eeg_raw.shape)
    print("Continuous filtered:", eeg_filtered.shape)
    print("Markers:", markers.shape)
    print("X_raw:", X_raw.shape)
    print("X_filtered:", X_filtered.shape)
    print("X_csp:", X_csp.shape)
    print("y:", y.shape)
    print("Left:", int((y == MARKERS["LEFT"]).sum()))
    print("Right:", int((y == MARKERS["RIGHT"]).sum()))

    file_path = os.path.join(subject_folder, f"{session_name}.npz")

    np.savez(
        file_path,
        continuous_eeg_raw=eeg_raw,
        continuous_eeg_filtered=eeg_filtered,
        continuous_markers=markers,
        X_raw=X_raw,
        X_filtered=X_filtered,
        X_csp=X_csp,
        y=y,
        fs=FS,
        eeg_channels=np.array(eeg_channels),
        marker_channel=marker_channel,
        marker_indices=kept_marker_indices,
        trial_order=np.array(trial_labels),
        fixation_sec=FIXATION_SEC,
        imagery_sec=IMAGERY_SEC,
        rest_sec=REST_SEC,
        warmup_sec=WARMUP_SEC,
        csp_start_after_cue=CSP_START_AFTER_CUE,
        csp_end_after_cue=CSP_END_AFTER_CUE,
        metadata=np.array(metadata, dtype=object),
    )

    print("\nSaved:")
    print(file_path)


if __name__ == "__main__":
    main()


Preparing OpenBCI session...
Warming up for 5s...

Starting recording.
Session: DEVIN_02
Fixation -> cue/imagery -> rest
Do not physically move.

Trial 1/30
Fixation: 2s
>>> THINK LEFT <<<
Rest: 3s
Trial 2/30
Fixation: 2s
>>> THINK RIGHT <<<
Rest: 3s
Trial 3/30
Fixation: 2s
>>> THINK RIGHT <<<
Rest: 3s
Trial 4/30
Fixation: 2s
>>> THINK RIGHT <<<
Rest: 3s
Trial 5/30
Fixation: 2s
>>> THINK RIGHT <<<
Rest: 3s
Trial 6/30
Fixation: 2s
>>> THINK RIGHT <<<
Rest: 3s
Trial 7/30
Fixation: 2s
>>> THINK RIGHT <<<
Rest: 3s
Trial 8/30
Fixation: 2s
>>> THINK LEFT <<<
Rest: 3s
Trial 9/30
Fixation: 2s
>>> THINK RIGHT <<<
Rest: 3s
Trial 10/30
Fixation: 2s
>>> THINK RIGHT <<<
Rest: 3s
Trial 11/30
Fixation: 2s
>>> THINK RIGHT <<<
Rest: 3s
Trial 12/30
Fixation: 2s
>>> THINK RIGHT <<<
Rest: 3s
Trial 13/30
Fixation: 2s
>>> THINK RIGHT <<<
Rest: 3s
Trial 14/30
Fixation: 2s
>>> THINK RIGHT <<<
Rest: 3s
Trial 15/30
Fixation: 2s
>>> THINK RIGHT <<<
Rest: 3s
Trial 16/30
Fixation: 2s
>>> THINK LEFT <<<
Rest: 3s
T

100%|██████████| 8/8 [00:00<00:00, 157.15it/s]


Final shapes:
Continuous raw: (8, 76594)
Continuous filtered: (8, 76594)
Markers: (76594,)
X_raw: (30, 8, 1750)
X_filtered: (30, 8, 1750)
X_csp: (30, 8, 750)
y: (30,)
Left: 15
Right: 15

Saved:
Datasets/Raw\DEVIN\DEVIN_02.npz
